# Sim2real gap — hardware drop telemetry

Assesses why the righting policy that scores 68–96% in simulation fails on the robot,
using the 7 logged drops in this folder.

Each stage is independent and prints its own verdict. They are ordered so that the
cheapest, most *eliminative* checks run first: a deployment bug makes every physics
question downstream meaningless, so identify the flown policy before modelling anything.

**Layout of a telemetry row** (`controller.py::main`, written after `set_motors`, so
the observation and the action it produced are on the same row):

| cols | meaning |
|---|---|
| `Time` | Pi wall clock, s |
| `F_Q0..3` | front IMU quaternion wxyz, **already IMU-aligned** — the exact object fed to `to_projected_gravity` |
| `F_M1`,`F_M2` | measured `rot1` (roll), `pitch`, rad |
| `F_ACC` | front accel magnitude, m/s^2 (free-fall trigger is `< 3.5`) |
| `Cmd_F1`,`Cmd_F2` | commanded `rot1`, `pitch` targets, **rad** (post joint-range mapping) |
| `B_Q0..3` | back IMU — **no rear IMU exists on this robot**; constant identity, ignored |
| `B_M1`,`B_M2` | measured `tail`, `rot2`, rad |
| `Cmd_B1`,`Cmd_B2` | commanded `tail`, `-rot1` targets, rad |

Rear-body pose is derived kinematically from the front IMU plus joint angles
(`reconstruct_viz.rear_world_quat`), which is the intended design, not a fallback.

## Stage 0 — Load and condition

Parses the CSVs, checks the control loop actually held 50 Hz, and finds the airborne
window from the accelerometer. The sim has contact disabled, so only pre-impact frames
are comparable to it — everything downstream is restricted to that window.

In [ ]:
import os, sys, glob, csv, json
import numpy as np
from scipy.spatial.transform import Rotation as R
import matplotlib
try:
    get_ipython()               # noqa: F821 -- Jupyter: keep the inline backend
except NameError:
    matplotlib.use("Agg")       # plain script: render headless
import matplotlib.pyplot as plt

CWD = os.path.abspath(os.getcwd())
REPO = os.path.dirname(CWD) if os.path.basename(CWD) == "telemetry" else CWD
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "hardware"))
TELEM = os.path.join(REPO, "telemetry")
os.chdir(REPO)      # cat_env resolves model/cat.xml relative to the cwd

import cat_env.env_util as util
from distillation import (stack_frames, N_FRAMES,
                          FRONT_GRAV_SLICE, JOINT_ANGLE_SLICE)
import reconstruct_viz as RV

# Joint-range mapping used by controller.py to turn a normalised action into rad.
ROLL_RANGE, PITCH_RANGE, TAIL_RANGE = 7.28, 1.57, 1.9199
LOOP_HZ = 50.0
UPRIGHT_DEG = 30.0        # cat_env / evaluate.py success threshold
FREE_FALL_ACC = 3.5       # controller.py drop trigger
JOINTS = ("rot1", "pitch", "rot2", "tail")

# Column indices, from controller.py::log.append
C = dict(t=0, fq=slice(1, 5), rot1=5, pitch=6, facc=7, cmd_rot1=8, cmd_pitch=9,
         bq=slice(10, 14), tail=14, rot2=15, bacc=16, cmd_tail=17, cmd_rot2=18)


def tilt_deg(quat_wxyz):
    """Angle between a body's +z and world +z, degrees. Matches evaluate.tilt_deg."""
    up = R.from_quat(quat_wxyz, scalar_first=True).apply([0, 0, 1])
    return np.degrees(np.arccos(np.clip(np.atleast_2d(up)[:, 2], -1.0, 1.0)))


def smooth_deriv(y, t, win=5):
    """Savitzky-Golay-style derivative: least-squares line over a centred window.

    Raw 50 Hz differencing of a fused quaternion is dominated by sensor noise, which
    inflates every velocity-derived quantity (body rate, joint rate, momentum). A
    linear fit over `win` samples suppresses that without introducing lag.
    """
    y = np.asarray(y, float); n = len(y)
    out = np.zeros_like(y)
    h = win // 2
    for i in range(n):
        lo, hi = max(0, i - h), min(n, i + h + 1)
        tt = t[lo:hi] - t[i]
        A = np.vstack([tt, np.ones_like(tt)]).T
        out[i] = np.linalg.lstsq(A, y[lo:hi], rcond=None)[0][0]
    return out


def load_run(path):
    rows = list(csv.reader(open(path)))[1:]
    d = np.array([[float(x) for x in r] for r in rows])
    t = d[:, C["t"]] - d[0, C["t"]]
    fq = d[:, C["fq"]]
    fq = fq / np.linalg.norm(fq, axis=1, keepdims=True)

    # Measured joints in SIM order [rot1, pitch, rot2, tail] (controller.py builds the
    # student frame in exactly this order).
    q = np.stack([d[:, C["rot1"]], d[:, C["pitch"]], d[:, C["rot2"]], d[:, C["tail"]]], 1)
    # Commanded targets, same order. Cmd_B2 is -rot1 by construction (counter-twist).
    cmd = np.stack([d[:, C["cmd_rot1"]], d[:, C["cmd_pitch"]],
                    -d[:, C["cmd_rot1"]], d[:, C["cmd_tail"]]], 1)
    # Normalised action the policy emitted, recovered by inverting the range map.
    act = np.stack([d[:, C["cmd_rot1"]] / ROLL_RANGE,
                    d[:, C["cmd_pitch"]] / PITCH_RANGE,
                    d[:, C["cmd_tail"]] / TAIL_RANGE], 1)

    name = os.path.basename(path)
    release = float(name.split("deg")[0])

    # Airborne window. F_ACC CANNOT be used to find the landing: the IMU sits off the
    # COM, so the robot's own rotation registers as 10-38 m/s^2 of centripetal/tangential
    # acceleration, and the peaks land mid-run (frames 10-24) rather than at the end.
    # The controller instead defines the episode itself -- it triggers on free-fall and
    # drives for CONTROL_DURATION = 0.74 s = 37 steps, which is exactly the sim episode
    # and exactly the number of rows logged. So the whole log IS the airborne window.
    acc = d[:, C["facc"]]
    t_imp = len(t)

    front_tilt = tilt_deg(fq)
    rear_q = np.array([RV.rear_world_quat(fq[i], q[i]) for i in range(len(t))])
    rear_tilt = tilt_deg(rear_q)

    return dict(name=name, path=path, release=release, t=t, fq=fq, q=q, cmd=cmd,
                act=act, acc=acc, n=len(t), t_impact=t_imp,
                front_tilt=front_tilt, rear_tilt=rear_tilt, rear_q=rear_q)


RUNS = [load_run(p) for p in sorted(glob.glob(os.path.join(TELEM, "*.csv")))]
print(f"{len(RUNS)} runs loaded from {TELEM}\n")

hdr = f"{'run':22} {'rel':>4} {'n':>3} {'dur':>6} {'dt ms':>12} {'acc max':>8} {'front tilt':>16} {'rear tilt':>16}"
print(hdr); print("-" * len(hdr))
for r in RUNS:
    dt = np.diff(r["t"]) * 1e3
    k = r["t_impact"] - 1
    print(f"{r['name']:22} {r['release']:>4.0f} {r['n']:>3d} {r['t'][-1]:>5.3f}s "
          f"{dt.mean():>6.1f}+-{dt.std():<4.1f} {r['acc'].max():>8.1f} "
          f"{r['front_tilt'][0]:>6.1f}->{r['front_tilt'][k]:<6.1f}  "
          f"{r['rear_tilt'][0]:>6.1f}->{r['rear_tilt'][k]:<6.1f}")

dts = np.concatenate([np.diff(r["t"]) for r in RUNS]) * 1e3
print(f"\nloop period over all runs: {dts.mean():.2f} +- {dts.std():.2f} ms "
      f"(target {1e3/LOOP_HZ:.1f} ms)  ->  timing is NOT a gap source")
print("rear IMU columns are constant identity by design (no rear IMU on this robot); "
      "rear pose above is forward-kinematic from the front IMU + joints.")

## Stage 1 — Which policy actually flew?  *(deployment parity)*

Every row carries the observation *and* the action it produced, and the logged front
quaternion is the IMU-aligned one that `to_projected_gravity` consumes. So the student
observation can be rebuilt exactly and replayed through each exported `.onnx`, then
compared against the commands that were really sent.

A mismatch here would invalidate every physics conclusion downstream, which is why this
runs first. `e2e_test.py` cannot catch this class of bug: it verifies the code path is
self-consistent, not that the intended *weights* were the ones deployed.

In [ ]:
import onnxruntime as ort

def replay_onnx(sess, inp, fq, q):
    """Rebuild the 14-dim student obs per row and run the policy, exactly as controller.py."""
    frames = [np.concatenate([util.to_projected_gravity(fq[i]), q[i]]) for i in range(len(fq))]
    hist, out = None, []
    for f in frames:
        hist = [f] * N_FRAMES if hist is None else hist[1:] + [f]   # prefill on tick 0
        x = stack_frames(hist).astype(np.float32).reshape(1, -1)
        out.append(sess.run(None, {inp: x})[0][0])
    return np.array(out)


cands = {}
for variant, fn in (("tail", "cat_controller.onnx"), ("notail", "cat_controller_notail.onnx")):
    p = os.path.join(REPO, "policies", fn)
    if os.path.exists(p):
        s = ort.InferenceSession(p)
        cands[variant] = (s, s.get_inputs()[0].name, fn)

print("max |replayed action - logged action|   (action is normalised, range [-1, 1])\n")
print(f"{'run':22} " + " ".join(f"{v:>12}" for v in cands))
print("-" * (22 + 13 * len(cands)))
match = {}
for r in RUNS:
    errs = {}
    for v, (s, inp, _) in cands.items():
        errs[v] = float(np.abs(replay_onnx(s, inp, r["fq"], r["q"]) - r["act"]).max())
    match[r["name"]] = min(errs, key=errs.get)
    print(f"{r['name']:22} " + " ".join(f"{errs[v]:>12.6f}" for v in cands))

winner = set(match.values())
print()
if len(winner) == 1 and max(
        float(np.abs(replay_onnx(*cands[list(winner)[0]][:2], r["fq"], r["q"]) - r["act"]).max())
        for r in RUNS) < 1e-4:
    v = list(winner)[0]
    print(f"VERDICT: every drop reproduces EXACTLY under '{v}' ({cands[v][2]}).")
    print("         The deployment path is bit-faithful -- the observation build, frame")
    print("         stacking and range mapping on the Pi all match the sim.")
    FLOWN = v
else:
    FLOWN = None
    print("VERDICT: no single policy reproduces the logs -- investigate before trusting anything below.")

## Stage 2 — The tail channel

`cat_notail.xml` does not delete the tail. It keeps the joint and starves its motor:

```xml
<motor name="motor4" joint="tail" ctrlrange="-1e-6 1e-6"/>
```

so in simulation `action[2]` moves the tail by <3e-7 rad — it is inert. A policy trained
there receives **no gradient signal on its third output**: every value scores identically,
so whatever it emits is arbitrary.

`controller.py` has no such clamp. It maps `action[2]` through the full `TAIL_RANGE`
(+-110 deg) and writes it to a real, working tail motor. This stage measures what that
untrained channel actually commanded.

In [ ]:
import gymnasium as gym, cat_env

print("Is action[2] inert in each sim variant?  (37 steps of a=[0,0,+1], max tail command)\n")
for env_id, label in (("Cat-v0", "tail"), ("CatNoTail-v0", "notail")):
    u = gym.make(env_id).unwrapped
    u.reset(seed=0); u.action_delay = 0; u.action_buffer = []
    q0 = u.data.qpos[7:].copy()
    for _ in range(37):
        u.step(np.array([0.0, 0.0, 1.0], dtype=np.float32))
    dq = u.data.qpos[7:] - q0
    ctrl = u.model.actuator_ctrlrange[3]
    print(f"  {label:7} tail motor ctrlrange {ctrl[0]:+.1e}..{ctrl[1]:+.1e}   "
          f"tail joint moved {abs(dq[3]):.2e} rad")
    u.close()

print(f"\nWhat the tail motor was actually commanded on hardware "
      f"(TAIL_RANGE = +-{TAIL_RANGE:.4f} rad = +-{np.degrees(TAIL_RANGE):.0f} deg):\n")
print(f"{'run':22} {'|a2| mean':>10} {'cmd range (rad)':>18} {'travel':>8} {'% of full swing':>16}")
print("-" * 78)
for r in RUNS:
    k = r["t_impact"]
    a2 = r["act"][:k, 2]
    ct = r["cmd"][:k, 3]
    travel = np.abs(np.diff(ct)).sum()
    print(f"{r['name']:22} {np.abs(a2).mean():>10.3f} "
          f"{ct.min():>+8.2f}..{ct.max():<+8.2f} {travel:>8.2f} "
          f"{100*(ct.max()-ct.min())/(2*TAIL_RANGE):>15.0f}%")

print("\nA channel with zero effect in training drove a real +-110 deg motor on the robot.")

## Stage 3 — Where did the body rotation come from?

In free fall there is no external torque, so total angular momentum **L** is conserved.
That makes L diagnostic of the *source* of the tumble:

- **|L| large and constant** — the rotation was imparted at release (a thrown robot).
- **|L| ~ 0 and constant** — the body rotation is entirely self-induced: the robot is
  counter-rotating against its own limbs, which is the righting mechanism itself, and
  also what a flailing untrained tail would produce.

L is computed by loading each measured state into MuJoCo (`mj_subtreeVel`) with the
nominal, un-randomised model, so the robot's real inertia tensor does the work. Velocities
use the smoothed derivative — raw 50 Hz differencing of a fused quaternion inflates |L|.

In [ ]:
import mujoco

u = gym.make("Cat-v0").unwrapped
u.reset(seed=0)
# Undo the episode's domain-randomisation draw so this is the nominal robot.
u.model.body_mass[:] = u.nominal_mass
u.model.body_ipos[:] = u.nominal_ipos
u.model.body_inertia[:] = u.nominal_inertia
M, D = u.model, u.data
mujoco.mj_setConst(M, D)     # refresh derived constants, else subtree inertia is stale
BID = mujoco.mj_name2id(M, mujoco.mjtObj.mjOBJ_BODY, "front_body")

# Reference scale: |L| produced by the largest initial tumble the sim ever trains on.
from cat_env.cat_env import init_ang_vel_max, init_joint_pos_max
D.qpos[:] = u.init_qpos; D.qvel[:] = 0
D.qvel[3:6] = init_ang_vel_max            # per-axis max
mujoco.mj_forward(M, D); mujoco.mj_subtreeVel(M, D)
L_SIM_MAX = np.linalg.norm(D.subtree_angmom[BID])
print(f"reference: |L| at the sim's MAXIMUM initial tumble "
      f"(qvel[3:6] = {init_ang_vel_max} rad/s each) = {L_SIM_MAX:.4f} kg m^2/s\n")


def momentum(r):
    t, fq, q = r["t"], r["fq"], r["q"]
    rs = R.from_quat(fq, scalar_first=True)
    # Body-frame angular velocity: successive relative rotations, smoothed.
    rv = np.zeros((len(t), 3))
    for i in range(len(t) - 1):
        rv[i] = (rs[i].inv() * rs[i + 1]).as_rotvec() / (t[i + 1] - t[i])
    rv[-1] = rv[-2]
    w = np.stack([np.convolve(rv[:, k], np.ones(5) / 5, mode="same") for k in range(3)], 1)
    jv = np.stack([smooth_deriv(q[:, k], t, 5) for k in range(4)], 1)
    L = np.zeros((len(t), 3))
    for i in range(len(t)):
        D.qpos[3:7] = fq[i]; D.qpos[7:] = q[i]
        D.qvel[0:3] = 0; D.qvel[3:6] = w[i]; D.qvel[6:] = jv[i]
        mujoco.mj_forward(M, D); mujoco.mj_subtreeVel(M, D)
        L[i] = D.subtree_angmom[BID]
    return L, w, jv


print(f"{'run':22} {'|L| mean':>10} {'|L| sd':>8} {'vs sim max':>11} {'|w0| rad/s':>11} {'verdict':>16}")
print("-" * 84)
MOM = {}
for r in RUNS:
    L, w, jv = momentum(r)
    MOM[r["name"]] = (L, w, jv)
    k = r["t_impact"]
    n = np.linalg.norm(L[:k], axis=1)
    w0 = np.linalg.norm(w[:3], axis=1).mean()
    ratio = n.mean() / L_SIM_MAX
    verdict = "release tumble" if ratio > 1.0 else ("borderline" if ratio > 0.5 else "self-induced")
    print(f"{r['name']:22} {n.mean():>10.4f} {n.std():>8.4f} {ratio:>10.2f}x "
          f"{w0:>11.2f} {verdict:>16}")

print("\n|L| is conserved in free fall, so a near-constant value is a consistency check on")
print("the data; the MAGNITUDE relative to the sim's maximum is what says whether the")
print("robot was thrown or is spinning itself up.")

## Stage 4 — Actuation gap: did the joints follow the commands?

The same quantity is available on both sides, computed identically:

- **hardware** — `Cmd_*` vs measured `*_M*` in this telemetry;
- **sim** — `cat_env` logs `self.ctrls` = `[mapped target(3), joint qpos(4)]` every step,
  which is what `plots/plot_joint_tracking.py` reads.

If the real joints lag the commands far more than the sim's do, the policy is flying a
plant that does not exist.

In [ ]:
def tracking_stats(cmd, meas, t):
    """Per-joint tracking error. The command at step k is the target for step k+1."""
    e = meas[1:] - cmd[:-1]
    return dict(rms=np.sqrt((e ** 2).mean(0)), peak=np.abs(e).max(0),
                cmd_travel=np.abs(np.diff(cmd, axis=0)).sum(0),
                meas_travel=np.abs(np.diff(meas, axis=0)).sum(0))


# --- hardware ---
print("HARDWARE  (rad)\n")
print(f"{'run':22} " + " ".join(f"{j:>17}" for j in JOINTS))
print(f"{'':22} " + " ".join(f"{'rms /  cmd->meas':>17}" for _ in JOINTS))
print("-" * 92)
hw_rms, hw_ratio = [], []
for r in RUNS:
    k = r["t_impact"]
    s = tracking_stats(r["cmd"][:k], r["q"][:k], r["t"][:k])
    hw_rms.append(s["rms"])
    hw_ratio.append(s["meas_travel"] / np.maximum(s["cmd_travel"], 1e-9))
    print(f"{r['name']:22} " + " ".join(
        f"{s['rms'][i]:>5.2f} /{s['cmd_travel'][i]:>6.1f}->{s['meas_travel'][i]:<5.1f}"
        for i in range(4)))
hw_rms = np.array(hw_rms); hw_ratio = np.array(hw_ratio)

# --- sim, same policy, same measure ---
u2 = gym.make("CatNoTail-v0" if FLOWN == "notail" else "Cat-v0").unwrapped
sim_rms, sim_ratio = [], []
for seed in range(8):
    obs, _ = u2.reset(seed=seed)
    u2.action_delay = 0; u2.action_buffer = []
    sess, inp, _ = cands[FLOWN]
    hist = None
    for _ in range(37):
        fr = np.concatenate([obs[FRONT_GRAV_SLICE], obs[JOINT_ANGLE_SLICE]])
        hist = [fr] * N_FRAMES if hist is None else hist[1:] + [fr]
        a = sess.run(None, {inp: stack_frames(hist).astype(np.float32).reshape(1, -1)})[0][0]
        obs, _, te, tr, _ = u2.step(a)
        if te or tr:
            break
    ctrls = np.array(u2.ctrls)                       # [rot1,pitch,tail targets | 4 qpos]
    cmd = np.stack([ctrls[:, 0], ctrls[:, 1], -ctrls[:, 0], ctrls[:, 2]], 1)
    meas = ctrls[:, 3:7]
    s = tracking_stats(cmd, meas, None)
    sim_rms.append(s["rms"]); sim_ratio.append(s["meas_travel"] / np.maximum(s["cmd_travel"], 1e-9))
sim_rms = np.array(sim_rms); sim_ratio = np.array(sim_ratio)

print(f"\n{'':22} " + " ".join(f"{j:>12}" for j in JOINTS))
print(f"{'RMS tracking error':22} " + " ".join(f"{v:>12.3f}" for v in hw_rms.mean(0)) + "   hardware")
print(f"{'':22} " + " ".join(f"{v:>12.3f}" for v in sim_rms.mean(0)) + "   sim")
print(f"{'':22} " + " ".join(f"{a/max(b,1e-9):>11.1f}x" for a, b in zip(hw_rms.mean(0), sim_rms.mean(0))) + "   ratio")
print(f"\n{'travel achieved/commanded':22}")
print(f"{'  hardware':22} " + " ".join(f"{v:>12.2f}" for v in hw_ratio.mean(0)))
print(f"{'  sim':22} " + " ".join(f"{v:>12.2f}" for v in sim_ratio.mean(0)))
print("\n1.0 = the joint went exactly as far as it was told. Below 1.0 = it could not keep up.")


# ---- inline: commanded vs measured angle, every joint of every run ----
JLIM = (ROLL_RANGE, PITCH_RANGE, ROLL_RANGE, TAIL_RANGE)
fig, axes = plt.subplots(len(RUNS), 4, figsize=(15, 2.05 * len(RUNS)), sharex=True)
for ri, r in enumerate(RUNS):
    k = r["t_impact"]
    for ji, j in enumerate(JOINTS):
        ax = axes[ri, ji]
        ax.axhline(JLIM[ji], c="0.8", lw=0.8); ax.axhline(-JLIM[ji], c="0.8", lw=0.8)
        ax.plot(r["t"][:k], r["cmd"][:k, ji], lw=1.3, c="tab:red", label="commanded")
        ax.plot(r["t"][:k], r["q"][:k, ji], lw=1.3, c="tab:blue", label="measured")
        ax.margins(y=0.12)
        if ri == 0:
            ax.set_title(j, fontsize=11)
        if ji == 0:
            ax.set_ylabel(r["name"].replace(".csv", "").replace("deg", "\u00b0 "), fontsize=7)
        if ri == len(RUNS) - 1:
            ax.set_xlabel("t (s)", fontsize=8)
        ax.tick_params(labelsize=6)
axes[0, 0].legend(fontsize=7, loc="upper left")
fig.suptitle("Commanded vs measured joint angle (rad).  Grey = joint limit.\n"
             "rot1/rot2 are commanded across the full range and barely move.", fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.965])
plt.savefig(os.path.join(TELEM, "stage4_tracking.png"), dpi=130)
plt.show()

# ---- inline: how much of the commanded travel each joint actually achieved ----
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
x = np.arange(4); w = 0.36
ax = axes[0]
ax.bar(x - w/2, np.array([np.abs(np.diff(r["cmd"][:r["t_impact"]], axis=0)).sum(0) for r in RUNS]).mean(0),
       w, label="commanded", color="tab:red")
ax.bar(x + w/2, np.array([np.abs(np.diff(r["q"][:r["t_impact"]], axis=0)).sum(0) for r in RUNS]).mean(0),
       w, label="achieved", color="tab:blue")
ax.set_xticks(x); ax.set_xticklabels(JOINTS); ax.set_ylabel("travel per drop (rad)")
ax.set_title("Hardware: commanded vs achieved travel"); ax.legend(fontsize=8)
ax = axes[1]
ax.bar(x - w/2, hw_ratio.mean(0), w, label="hardware", color="tab:blue")
ax.bar(x + w/2, sim_ratio.mean(0), w, label="sim", color="tab:green")
ax.axhline(1.0, ls="--", c="k", lw=1)
ax.set_xticks(x); ax.set_xticklabels(JOINTS); ax.set_ylim(0, 1.1)
ax.set_ylabel("achieved / commanded"); ax.set_title("Tracking ratio (1.0 = kept up)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(TELEM, "stage4_travel.png"), dpi=130)
plt.show()

## Stage 5 — Was the drop inside the training distribution?

`cat_env.reset_model` samples the release state as `qvel[3:6] ~ U(-0.5, 0.5)` per axis and
joint angles within `+-0.2` rad. A drop outside that support is one the policy has never
seen, which is an out-of-distribution failure rather than a modelling error — and the
cheapest thing in the project to fix.

In [ ]:
w_max = init_ang_vel_max * np.sqrt(3)     # |w| bound of the sim's uniform cube
print(f"sim initial tumble: per-axis U(-{init_ang_vel_max}, {init_ang_vel_max}) rad/s "
      f"-> |w0| <= {w_max:.2f} rad/s")
print(f"sim initial joints: U(-{init_joint_pos_max}, {init_joint_pos_max}) rad\n")

print(f"{'run':22} {'|w0|':>7} {'/ sim max':>10} {'max |q0|':>9} {'/ sim max':>10} {'tilt0':>7} {'in support?':>12}")
print("-" * 82)
n_out = 0
for r in RUNS:
    L, w, jv = MOM[r["name"]]
    w0 = np.linalg.norm(w[:3], axis=1).mean()
    q0 = np.abs(r["q"][0]).max()
    ok = (w0 <= w_max) and (q0 <= init_joint_pos_max)
    n_out += (not ok)
    print(f"{r['name']:22} {w0:>7.2f} {w0/w_max:>9.1f}x {q0:>9.2f} "
          f"{q0/init_joint_pos_max:>9.1f}x {r['front_tilt'][0]:>7.1f} {'yes' if ok else 'NO':>12}")

print(f"\n{n_out}/{len(RUNS)} drops start outside the sim's training support.")
print("Note the joint-angle column: the robot is released from wherever the previous run")
print("left it, whereas the sim always starts within +-0.2 rad of the home pose.")

## Stage 6 — What the network actually saw

Stages 1-5 ask whether the robot did the right thing. This one asks whether the policy
was ever given a fair question: the student is a function of 14 numbers, and if those
numbers land outside the range it was distilled on, its output is extrapolation.

The observation is rebuilt from raw sensor readings exactly as `controller.py` does, and
compared against the sim observations the student was trained and evaluated against --
generated at the **same release attitudes** (roll 180/90/45, pitch 0) and through the
same sensor-noise model, so the comparison isolates the plant rather than the protocol.

`stack_frames` layout for `N_FRAMES = 2` is `[grav(3), d_grav(3), joints(4), d_joints(4)]`.
The `d_` half is the velocity-like signal the 2-frame stack exists to supply, and it is
where Stage 3 (too much body rotation) and Stage 4 (too little joint motion) both land.

In [ ]:
sys.path.insert(0, os.path.join(REPO, "docs"))
import evaluate as EV
from distillation import sample_sensor_bias, get_noisy_student_frame

OBS_LABELS = ([f"grav_{c}" for c in "xyz"] + [f"d_grav_{c}" for c in "xyz"]
              + [f"q_{j}" for j in JOINTS] + [f"dq_{j}" for j in JOINTS])


def obs_from_frames(frames):
    """Frame sequence -> the stacked observations the net saw, one row per tick."""
    hist, out = None, []
    for f in frames:
        hist = [f] * N_FRAMES if hist is None else hist[1:] + [f]
        out.append(stack_frames(hist))
    return np.array(out)


# ---- hardware: rebuilt from the logged IMU + encoders ----
HW_OBS = {}
for r in RUNS:
    k = r["t_impact"]
    frames = [np.concatenate([util.to_projected_gravity(r["fq"][i]), r["q"][i]])
              for i in range(k)]
    HW_OBS[r["name"]] = obs_from_frames(frames)
hw = np.concatenate(list(HW_OBS.values()))

# ---- sim: same policy, same release attitudes, same noise model ----
env_o = gym.make("CatNoTail-v0" if FLOWN == "notail" else "Cat-v0")
u_o = env_o.unwrapped
sess, inp, _ = cands[FLOWN]
rolls = sorted({int(r["release"]) for r in RUNS})
sim_obs, EPISODES = [], 40
np.random.seed(0)
for roll in rolls:
    for ep in range(EPISODES):
        obs, _ = env_o.reset()
        obs = EV.force_attitude(u_o, float(roll), None)     # pitch 0, DR/tumble still vary
        bias = sample_sensor_bias()
        hist, done = None, False
        while not done:
            fr = get_noisy_student_frame(obs, bias)
            hist = [fr] * N_FRAMES if hist is None else hist[1:] + [fr]
            x = stack_frames(hist)
            sim_obs.append(x)
            a = sess.run(None, {inp: x.astype(np.float32).reshape(1, -1)})[0][0]
            obs, _, te, tr, _ = env_o.step(a)
            done = te or tr
sim = np.array(sim_obs)
print(f"hardware: {hw.shape[0]} obs from {len(RUNS)} drops")
print(f"sim:      {sim.shape[0]} obs from {len(rolls)*EPISODES} drops at roll "
      + "/".join(map(str, rolls)) + ", same noise model\n")

lo, hi = np.percentile(sim, 1, axis=0), np.percentile(sim, 99, axis=0)
outside = ((hw < lo) | (hw > hi)).mean(0) * 100
print(f"{'channel':>11} {'sim p1..p99':>20} {'hw min..max':>20} {'sd hw/sim':>10} {'% hw outside':>13}")
print("-" * 80)
for i, lab in enumerate(OBS_LABELS):
    print(f"{lab:>11} {lo[i]:>9.2f}..{hi[i]:<9.2f} {hw[:, i].min():>9.2f}..{hw[:, i].max():<9.2f} "
          f"{hw[:, i].std()/max(sim[:, i].std(), 1e-9):>9.2f}x {outside[i]:>12.1f}%")

grp = {"grav": slice(0, 3), "d_grav": slice(3, 6), "joints": slice(6, 10), "d_joints": slice(10, 14)}
print()
for g, sl in grp.items():
    print(f"  {g:9} mean |value|  hw {np.abs(hw[:, sl]).mean():.3f}  vs sim {np.abs(sim[:, sl]).mean():.3f}"
          f"   ({np.abs(hw[:, sl]).mean()/max(np.abs(sim[:, sl]).mean(),1e-9):.2f}x)"
          f"   {outside[sl].mean():.1f}% of samples outside sim p1-p99")


# ---- per-channel distributions ----
fig, axes = plt.subplots(4, 4, figsize=(15, 9))
for i, lab in enumerate(OBS_LABELS):
    ax = axes[i // 4, i % 4]
    b = np.linspace(min(sim[:, i].min(), hw[:, i].min()), max(sim[:, i].max(), hw[:, i].max()), 60)
    ax.hist(sim[:, i], bins=b, density=True, alpha=0.55, color="tab:green", label="sim")
    ax.hist(hw[:, i], bins=b, density=True, histtype="step", lw=1.6, color="tab:blue", label="hardware")
    ax.axvline(lo[i], c="0.5", ls=":", lw=1); ax.axvline(hi[i], c="0.5", ls=":", lw=1)
    ax.set_title(f"{lab}   ({outside[i]:.0f}% out)", fontsize=9)
    ax.tick_params(labelsize=6); ax.set_yticks([])
for j in range(len(OBS_LABELS), 16):
    axes[j // 4, j % 4].axis("off")
axes[0, 0].legend(fontsize=7)
fig.suptitle("Student observation distribution: hardware vs sim (dotted = sim 1st/99th pct)", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(os.path.join(TELEM, "stage6_obs_dist.png"), dpi=130)
plt.show()


# ---- the two velocity-like magnitudes, over the drop ----
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (sl, name) in zip(axes, ((grp["d_grav"], "||d_grav||"), (grp["d_joints"], "||d_joints||"))):
    T = min(37, sim.shape[0])
    per_ep = sim[:, sl].reshape(-1, 1)
    smag = np.linalg.norm(sim[:, sl], axis=1)
    band = np.percentile(smag, [5, 50, 95])
    ax.axhspan(band[0], band[2], color="tab:green", alpha=0.2, label="sim 5-95 pct")
    ax.axhline(band[1], color="tab:green", lw=1.2, label="sim median")
    for r in RUNS:
        o = HW_OBS[r["name"]]
        ax.plot(r["t"][:len(o)], np.linalg.norm(o[:, sl], axis=1), lw=1.1,
                label=r["name"].replace(".csv", ""))
    ax.set_xlabel("t (s)"); ax.set_ylabel(name); ax.set_title(f"{name} per tick")
axes[0].legend(fontsize=6, ncol=2)
plt.tight_layout()
plt.savefig(os.path.join(TELEM, "stage6_obs_velocity.png"), dpi=130)
plt.show()

## Stage 7 — Dynamics gap: same commands, does the sim move the same way?

Replays each run's **logged command sequence** through the nominal (un-randomised) sim
from that run's measured release state, then compares two things:

1. **joints** — if the sim's joints follow the commands but the robot's did not, the gap is
   actuation (Stage 4);
2. **attitude** — if the joints agree but the body attitude diverges, the gap is in the
   physics: inertia, mass distribution, or momentum coupling the model gets wrong.

This is the test that separates "the robot did not do what it was told" from "it did, and
the world responded differently".

In [ ]:
def sim_replay(r, env):
    u = env.unwrapped
    u.reset(seed=0)
    u.model.body_mass[:] = u.nominal_mass
    u.model.body_ipos[:] = u.nominal_ipos
    u.model.body_inertia[:] = u.nominal_inertia
    mujoco.mj_setConst(u.model, u.data)
    for i, (kp, kd) in enumerate(u.pd_nominal):
        u.pd[i].kp, u.pd[i].kd = kp, kd
    u.action_delay = 0; u.action_buffer = []

    L, w, jv = MOM[r["name"]]
    qpos = u.init_qpos.copy(); qvel = u.init_qvel.copy()
    qpos[3:7] = r["fq"][0]          # measured release attitude
    qpos[7:] = r["q"][0]            # measured release joint angles
    qvel[3:6] = w[0]                # measured release body rate (body frame)
    qvel[6:] = jv[0]
    u.set_state(qpos, qvel)

    k = r["t_impact"]
    fq_sim, q_sim = [], []
    for i in range(k):
        u.step(r["act"][i].astype(np.float32))
        fq_sim.append(u.data.qpos[3:7].copy())
        q_sim.append(u.data.qpos[7:].copy())
    return np.array(fq_sim), np.array(q_sim)


env = gym.make("CatNoTail-v0" if FLOWN == "notail" else "Cat-v0")
print(f"{'run':22} {'joint err (rad)':>32} {'attitude err (deg)':>22}")
print(f"{'':22} " + " ".join(f"{j:>7}" for j in JOINTS) + f"    {'mid':>8} {'final':>8} {'tilt sim/real':>14}")
print("-" * 96)
for r in RUNS:
    fq_s, q_s = sim_replay(r, env)
    k = len(fq_s)
    jerr = np.abs(q_s - r["q"][:k]).mean(0)
    rs_s = R.from_quat(fq_s, scalar_first=True)
    rs_r = R.from_quat(r["fq"][:k], scalar_first=True)
    aerr = np.degrees([np.linalg.norm((rs_s[i].inv() * rs_r[i]).as_rotvec()) for i in range(k)])
    ts, tr = tilt_deg(fq_s)[-1], r["front_tilt"][k - 1]
    print(f"{r['name']:22} " + " ".join(f"{v:>7.2f}" for v in jerr)
          + f"    {aerr[k//2]:>8.1f} {aerr[-1]:>8.1f} {ts:>6.0f}/{tr:<6.0f}")

print("\nJoint columns near zero + large attitude error  => physics gap.")
print("Large joint columns                              => actuation gap (Stage 4 dominates).")

## Stage 8 — Outcome summary

Final tilts against the sim's own success threshold, plus the one statistical statement
7 drops can support. Per-angle success rates are **not** estimable at this sample size;
the aggregate is.

In [ ]:
# Sim reference for the flown policy, roll-conditioned (docs: student, 300 drops/angle).
SIM_RATE = {"tail":   {180: 0.683, 90: 0.950, 45: 0.963, 0: 0.960},
            "notail": {180: 0.187, 90: 0.680, 45: 0.777, 0: 0.850}}[FLOWN]

print(f"flown policy: {FLOWN}   (sim student rates: "
      + ", ".join(f"r{k}={100*v:.1f}%" for k, v in SIM_RATE.items()) + ")\n")
print(f"{'run':22} {'release':>8} {'front':>7} {'rear':>7} {'both<30':>9} {'sim P(success)':>15}")
print("-" * 74)
succ, p_list = 0, []
for r in RUNS:
    k = r["t_impact"] - 1
    f_, b_ = r["front_tilt"][k], r["rear_tilt"][k]
    ok = (f_ < UPRIGHT_DEG) and (b_ < UPRIGHT_DEG)
    succ += ok
    p = SIM_RATE.get(int(r["release"]), 0.5); p_list.append(p)
    print(f"{r['name']:22} {r['release']:>8.0f} {f_:>7.1f} {b_:>7.1f} {str(ok):>9} {100*p:>14.1f}%")

exp = sum(p_list)
print(f"\nobserved {succ}/{len(RUNS)} upright   |   sim expects {exp:.1f}/{len(RUNS)}")

# P(<= observed successes) under the sim's per-drop rates, by exact convolution.
dist = np.array([1.0])
for p in p_list:
    dist = np.convolve(dist, [1 - p, p])
pval = dist[:succ + 1].sum()
print(f"P(<= {succ} successes | sim rates) = {pval:.2e}")
print("\nSmall p => the aggregate gap is real even at n=7. Per-angle rates are not")
print("estimable here (3 drops at 180, 3 at 90, 1 at 45, none at 0).")

## Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

ax = axes[0]
for r in RUNS:
    k = r["t_impact"]
    ax.plot(r["t"][:k], r["front_tilt"][:k], label=r["name"].replace(".csv", ""))
ax.axhline(UPRIGHT_DEG, ls="--", c="k", lw=1, label=f"{UPRIGHT_DEG:.0f} deg threshold")
ax.set_xlabel("t (s)"); ax.set_ylabel("front tilt (deg)"); ax.set_title("Righting trajectories")
ax.legend(fontsize=6)

ax = axes[1]
for r in RUNS:
    k = r["t_impact"]
    ax.plot(r["t"][:k], r["cmd"][:k, 3], lw=1.2)
    ax.plot(r["t"][:k], r["q"][:k, 3], lw=1.2, ls="--")
ax.axhline(TAIL_RANGE, c="r", lw=0.8); ax.axhline(-TAIL_RANGE, c="r", lw=0.8)
ax.set_xlabel("t (s)"); ax.set_ylabel("tail angle (rad)")
ax.set_title("Tail: commanded (solid) vs measured (dashed)\nuntrained channel, real motor")

ax = axes[2]
w0s = [np.linalg.norm(MOM[r["name"]][1][:3], axis=1).mean() for r in RUNS]
ax.bar(range(len(RUNS)), w0s)
ax.axhline(w_max, c="r", ls="--", label=f"sim max |w0| = {w_max:.2f}")
ax.set_xticks(range(len(RUNS)))
ax.set_xticklabels([r["name"].replace(".csv", "") for r in RUNS], rotation=45, ha="right", fontsize=6)
ax.set_ylabel("|w0| (rad/s)"); ax.set_title("Release tumble vs training support"); ax.legend()

plt.tight_layout()
out = os.path.join(TELEM, "sim2real_summary.png")
plt.savefig(out, dpi=130)
plt.show()
print("saved", out)

## Stage 9 - Matched-condition case study: 180deg_fail1

One drop, replayed in sim from an identical release state -- attitude, body rate, joint
angles and joint rates all taken from the telemetry, on the nominal (un-randomised)
plant -- with the same student network closing the loop.

Both runs start from the same state and use the same weights, so **identical observations
would force identical actions**. Every divergence below is the plant, and the point at
which the traces separate says which channel carries it.

In [ ]:
CASE = "180deg_fail1.csv"
case = next(r for r in RUNS if r["name"] == CASE)
L_c, w_c, jv_c = MOM[CASE]

env_c = gym.make("CatNoTail-v0" if FLOWN == "notail" else "Cat-v0")
u_c = env_c.unwrapped
u_c.reset(seed=0)

# Nominal plant: this is a like-for-like trajectory comparison, so the episode's
# domain-randomisation draw is undone rather than left as an extra difference.
u_c.model.body_mass[:] = u_c.nominal_mass
u_c.model.body_ipos[:] = u_c.nominal_ipos
u_c.model.body_inertia[:] = u_c.nominal_inertia
u_c.model.dof_damping[:] = u_c.nominal_damping
u_c.model.dof_armature[:] = u_c.nominal_armature
u_c.model.dof_frictionloss[:] = u_c.nominal_frictionloss
u_c.model.actuator_ctrlrange[:] = u_c.nominal_ctrlrange
for i, (kp, kd) in enumerate(u_c.pd_nominal):
    u_c.pd[i].kp, u_c.pd[i].kd = kp, kd
    u_c.pd[i].meas_pos = case["q"][0][i]
mujoco.mj_setConst(u_c.model, u_c.data)
u_c.action_delay = 0
u_c.action_buffer = []

# Identical release condition, taken from the telemetry:
#   attitude + body rate from the front IMU, joint angles + rates from the encoders.
qpos = u_c.init_qpos.copy()
qvel = u_c.init_qvel.copy()
qpos[3:7] = case["fq"][0]
qpos[7:] = case["q"][0]
qvel[3:6] = w_c[0]          # body-frame angular velocity, matching MuJoCo's free joint
qvel[6:] = jv_c[0]
u_c.set_state(qpos, qvel)

print(f"matched initial condition from {CASE}")
print(f"  attitude   front tilt {case['front_tilt'][0]:.1f} deg   quat {np.round(case['fq'][0], 3)}")
print(f"  body rate  |w0| {np.linalg.norm(w_c[0]):.2f} rad/s   {np.round(w_c[0], 2)}")
print(f"  joints     {np.round(case['q'][0], 3)} rad")
print(f"  joint rate {np.round(jv_c[0], 3)} rad/s")

# Closed-loop student rollout. Clean frames: the sim's own sensors are exact, so this is
# the sim's best-faith reproduction of the drop rather than a re-run of distillation noise.
sess, inp, _ = cands[FLOWN]
n_steps = len(case["t"])
obs = u_c._get_obs()
hist, sim_obs_t, sim_act_t, sim_fq = None, [], [], []
for i in range(n_steps):
    fr = np.concatenate([obs[FRONT_GRAV_SLICE], obs[JOINT_ANGLE_SLICE]])
    hist = [fr] * N_FRAMES if hist is None else hist[1:] + [fr]
    x = stack_frames(hist)
    a = sess.run(None, {inp: x.astype(np.float32).reshape(1, -1)})[0][0]
    sim_obs_t.append(x)
    sim_act_t.append(a)
    sim_fq.append(u_c.data.qpos[3:7].copy())
    obs, _, te, tr, _ = u_c.step(a)
    if te or tr:
        break
sim_obs_t = np.array(sim_obs_t)
sim_act_t = np.array(sim_act_t)
sim_fq = np.array(sim_fq)

hw_obs_t = HW_OBS[CASE]
hw_act_t = case["act"]
k = min(len(sim_obs_t), len(hw_obs_t))
t = case["t"][:k]
print(f"\nrolled {len(sim_obs_t)} sim steps vs {len(hw_obs_t)} hardware steps; comparing {k}")
print(f"final front tilt:  sim {tilt_deg(sim_fq)[-1]:.1f} deg   hardware {case['front_tilt'][k-1]:.1f} deg")

# ---- action space, one subplot per dimension ----
ACT_LABELS = ("a0  roll", "a1  pitch", "a2  tail (untrained in no-tail)")
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for i, lab in enumerate(ACT_LABELS):
    ax = axes[i]
    ax.plot(t, hw_act_t[:k, i], lw=1.5, c="tab:blue", label="hardware")
    ax.plot(t, sim_act_t[:k, i], lw=1.5, c="tab:green", label="sim")
    ax.axhline(1, c="0.8", lw=0.8); ax.axhline(-1, c="0.8", lw=0.8)
    ax.set_title(lab, fontsize=10); ax.set_xlabel("t (s)"); ax.set_ylim(-1.15, 1.15)
    ax.tick_params(labelsize=7)
axes[0].set_ylabel("normalised action")
axes[0].legend(fontsize=8)
fig.suptitle(f"Action space over time -- sim vs hardware, identical release condition ({CASE})",
             fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.9])
plt.savefig(os.path.join(TELEM, "stage9_actions.png"), dpi=130)
plt.show()

# ---- observation space, one subplot per dimension ----
fig, axes = plt.subplots(4, 4, figsize=(15, 9))
for i, lab in enumerate(OBS_LABELS):
    ax = axes[i // 4, i % 4]
    ax.plot(t, hw_obs_t[:k, i], lw=1.4, c="tab:blue", label="hardware")
    ax.plot(t, sim_obs_t[:k, i], lw=1.4, c="tab:green", label="sim")
    div = np.abs(hw_obs_t[:k, i] - sim_obs_t[:k, i]).mean()
    ax.set_title(f"{lab}   mean|diff| {div:.3f}", fontsize=9)
    ax.tick_params(labelsize=6)
    if i // 4 == 3:
        ax.set_xlabel("t (s)", fontsize=8)
for j in range(len(OBS_LABELS), 16):
    axes[j // 4, j % 4].axis("off")
axes[0, 0].legend(fontsize=7)
fig.suptitle(f"Observation space over time -- sim vs hardware, identical release condition ({CASE})",
             fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(os.path.join(TELEM, "stage9_observations.png"), dpi=130)
plt.show()

# ---- when do the two runs stop being the same episode? ----
print(f"\n{'t (s)':>7} {'|d obs|':>9} {'|d act|':>9} {'attitude err (deg)':>19}")
rs_s = R.from_quat(sim_fq[:k], scalar_first=True)
rs_h = R.from_quat(case["fq"][:k], scalar_first=True)
for i in range(0, k, max(1, k // 8)):
    ae = np.degrees(np.linalg.norm((rs_s[i].inv() * rs_h[i]).as_rotvec()))
    print(f"{t[i]:>7.3f} {np.linalg.norm(hw_obs_t[i] - sim_obs_t[i]):>9.3f} "
          f"{np.linalg.norm(hw_act_t[i] - sim_act_t[i]):>9.3f} {ae:>19.1f}")
print("\nThe two runs share an initial state and the same network, so any divergence is the")
print("plant: identical observations would force identical actions.")
